In [9]:
import numpy as np
from collections import defaultdict

# ==========================================
# 1. 유틸리티 함수 (greedy_probs)
# ==========================================
def greedy_probs(Q, state, epsilon=0, action_size=4):
    qs = [Q[(state, a)] for a in range(action_size)]
    max_q = np.max(qs)
    
    base_prob = epsilon / action_size
    action_probs = {a: base_prob for a in range(action_size)}
    
    # Q값이 최대인 행동들(여러 개일 경우 고려) 찾기
    best_actions = [a for a, q in enumerate(qs) if q == max_q]
    
    # 최대 가치를 가진 행동에 남은 확률 몰아주기
    for action in best_actions:
        action_probs[action] += (1 - epsilon) / len(best_actions)
    
    return action_probs

# ==========================================
# 2. GridWorld 환경 클래스 (수정됨)
# ==========================================
class GridWorld:
    def __init__(self):
        self.action_space = [0, 1, 2, 3]
        self.action_meaning = {0: 'up', 1: 'down', 2: 'left', 3: 'right'}
        self.reward_map = np.array([
            [0, 0, 0, 1.0],
            [0, None, 0, -1.0],
            [0, 0, 0, 0]
        ])
        self.goal_state = (0, 3)
        self.wall_state = (1, 1)
        self.start_state = (2, 0)
        self.agent_state = self.start_state
        self.height_val = len(self.reward_map)
        self.width_val = len(self.reward_map[0])

    def reset(self):
        self.agent_state = self.start_state
        return self.agent_state

    def step(self, action):
        action_move_map = [(-1, 0), (1, 0), (0, -1), (0, 1)] # 상, 하, 좌, 우
        move = action_move_map[action]
        next_state = (self.agent_state[0] + move[0], self.agent_state[1] + move[1])
        ny, nx = next_state
        
        # 맵 밖으로 나가거나 벽인 경우 제자리
        if nx < 0 or nx >= self.width_val or ny < 0 or ny >= self.height_val:
            next_state = self.agent_state
        elif next_state == self.wall_state:
            next_state = self.agent_state

        self.agent_state = next_state
        
        # 보상 및 종료 여부 확인
        reward = self.reward_map[next_state[0], next_state[1]]
        if reward is None: reward = 0 # 벽 근처 등 None일 경우 0 처리
        
        done = (next_state == self.goal_state)
        
        return next_state, reward, done

    def render_q(self, Q):
        print("\n--- 학습된 Q-Values (Max Q) ---")
        for h in range(self.height_val):
            for w in range(self.width_val):
                state = (h, w)
                if state == self.wall_state:
                    print("  ###   ", end="")
                    continue
                if state == self.goal_state:
                    print("  GOAL  ", end="")
                    continue
                
                # 해당 상태에서 가장 높은 Q값 출력
                qs = [Q[(state, a)] for a in range(4)]
                max_q = np.max(qs) if qs else 0
                print(f"{max_q:7.2f} ", end="")
            print()
        print("-------------------------------\n")

# ==========================================
# 3. Q-Learning Agent (들여쓰기 수정됨)
# ==========================================
class QLearningAgent:
    def __init__(self):
        self.gamma = 0.9
        self.alpha = 0.8
        self.epsilon = 0.1
        self.action_size = 4
        
        random_actions = {0: 0.25, 1: 0.25, 2: 0.25, 3: 0.25}
        self.pi = defaultdict(lambda: random_actions)
        self.b = defaultdict(lambda: random_actions) # 행동 정책
        self.Q = defaultdict(lambda: 0)
    
    def get_action(self, state):
        action_probs = self.b[state] # 행동 정책에서 가져옴
        actions = list(action_probs.keys())
        probs = list(action_probs.values())
        return np.random.choice(actions, p=probs)
    
    def update(self, state, action, reward, next_state, done):
        if done: # 목표에 도달
            next_q_max = 0
        else: # 그 외에는 다음 상태에서 Q 함수의 최댓값 계산
            next_qs = [self.Q[next_state, a] for a in range(self.action_size)]
            next_q_max = np.max(next_qs)
        
        # Q 함수 갱신
        target = reward + self.gamma * next_q_max
        self.Q[state, action] += (target - self.Q[state, action]) * self.alpha
        
        # 행동 정책과 대상 정책 갱신
        self.pi[state] = greedy_probs(self.Q, state, epsilon=0)
        self.b[state] = greedy_probs(self.Q, state, self.epsilon)

# ==========================================
# 4. 실행 코드
# ==========================================
env = GridWorld()
agent = QLearningAgent()

episodes = 10000 
for episode in range(episodes):
    state = env.reset()
    
    while True:
        action = agent.get_action(state)
        next_state, reward, done = env.step(action)
        
        agent.update(state, action, reward, next_state, done)
        
        if done:
            break
        
        state = next_state

# 학습 결과 출력
print(f"총 {episodes} 에피소드 학습 완료")
env.render_q(agent.Q)

총 10000 에피소드 학습 완료

--- 학습된 Q-Values (Max Q) ---
   0.81    0.90    1.00   GOAL  
   0.73   ###      0.90    1.00 
   0.66    0.73    0.81    0.73 
-------------------------------

